In [4]:

import re
from pathlib import Path

import pandas as pd

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


ENV = "TEST"

YEAR_MIN, YEAR_MAX = 2000, 2025

FAMILY_TIME_COLUMN = "earliest_filing_year"
APPLN_TIME_COLUMN = "appln_filing_year"

TRUNCATED_YEARS = {
    "appln_filing_year": 3,
    "earliest_filing_year": 3,
    "earliest_publn_year": 1,
}

LAST_COMPLETE_YEAR = YEAR_MAX - max(
    TRUNCATED_YEARS[FAMILY_TIME_COLUMN],
    TRUNCATED_YEARS[APPLN_TIME_COLUMN],
)

IPR_TYPE = "PI"

INPUT_FILE = Path("outputs/step0/step0_sector_family_ids.csv")

OUTPUT_DIR = Path("outputs/step1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


PATENT_STOP_WORDS = {
    "said", "wherein", "whereby", "thereof", "therein",
    "comprising", "comprises", "comprise", "including",
    "includes", "plurality", "least", "configured",
    "provided", "providing", "present", "invention",
    "disclosed", "disclosure", "embodiment", "embodiments",
    "method", "methods", "device", "devices", "apparatus",
    "system", "systems", "unit", "first", "second",
    "third", "one", "having", "based",
}

ALL_STOP_WORDS = ENGLISH_STOP_WORDS | PATENT_STOP_WORDS


def strip_tags(text):
    return re.sub(
        r"\s+",
        " ",
        re.sub(r"<[^>]+>", " ", str(text))
    ).strip()


def clean_patent_text(text):
    if pd.isna(text) or not str(text).strip():
        return ""

    text = strip_tags(text).lower()

    words = re.findall(r"[a-z][a-z\-]+", text)

    return " ".join(
        word for word in words
        if word not in ALL_STOP_WORDS
    )


def harmonize_applicant(name):
    if pd.isna(name) or not str(name).strip():
        return "UNKNOWN"

    cleaned = re.sub(r"\s+", " ", str(name)).strip()

    return cleaned.upper()


def load_family_ids(input_file=INPUT_FILE):

    if not input_file.exists():
        raise FileNotFoundError(
            f"Input file not found: {input_file}. "
            "Run 01_extract.py first."
        )

    sector_ids_df = pd.read_csv(input_file)

    if "docdb_family_id" not in sector_ids_df.columns:
        raise ValueError(
            "The input CSV must contain a 'docdb_family_id' column."
        )

    family_ids = (
        pd.to_numeric(
            sector_ids_df["docdb_family_id"],
            errors="coerce",
        )
        .dropna()
        .drop_duplicates()
        .astype("int64")
        .tolist()
    )

    if not family_ids:
        raise ValueError(
            "No valid DOCDB family IDs found in the input CSV."
        )

    print(f"Loaded {len(family_ids):,} unique DOCDB families.")

    return family_ids


def run_cleaning(family_ids, db, patstat_client):

    if not family_ids:
        raise ValueError("family_ids cannot be empty.")

    print("\nPatent metadata extraction:")

    q_families = (
        db.query(
            TLS201_APPLN.docdb_family_id,
            TLS201_APPLN.appln_id,
            TLS201_APPLN.appln_auth,
            TLS201_APPLN.appln_filing_year,
            TLS201_APPLN.earliest_filing_year,
            TLS201_APPLN.docdb_family_size,
            TLS201_APPLN.nb_citing_docdb_fam,
            TLS202_APPLN_TITLE.appln_title,
            TLS203_APPLN_ABSTR.appln_abstract,
            TLS203_APPLN_ABSTR.appln_abstract_lg,
        )
        .filter(TLS201_APPLN.docdb_family_id.in_(family_ids))
        .filter(TLS201_APPLN.ipr_type == IPR_TYPE)
        .filter(
            TLS201_APPLN.earliest_filing_year.between(
                YEAR_MIN, YEAR_MAX
            )
        )
        .outerjoin(
            TLS202_APPLN_TITLE,
            TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
        )
        .outerjoin(
            TLS203_APPLN_ABSTR,
            TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
        )
    )

    df_raw = pd.DataFrame(patstat_client.df(q_families))

    if df_raw.empty:
        raise ValueError(
            "PATSTAT returned no patent records for the selected families."
        )

    print(f"Retrieved {len(df_raw):,} patent application records.")

    df_raw["is_en_abstract"] = (
        (df_raw["appln_abstract_lg"] == "en")
        & df_raw["appln_abstract"].notna()
        & (df_raw["appln_abstract"].fillna("").str.strip() != "")
    )

    df_families = (
        df_raw
        .sort_values(
            by=[
                "docdb_family_id",
                "is_en_abstract",
                "appln_filing_year",
            ],
            ascending=[True, False, True],
            na_position="last",
        )
        .drop_duplicates(
            subset=["docdb_family_id"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    df_families["is_truncated"] = (
        df_families[FAMILY_TIME_COLUMN] > LAST_COMPLETE_YEAR
    )

    df_families["abstract_en"] = (
        df_families["appln_abstract"]
        .where(df_families["is_en_abstract"], "")
        .fillna("")
    )

    df_families["clean_abstract"] = (
        df_families["abstract_en"]
        .apply(clean_patent_text)
    )

    print(
        f"Cleaned patent dataset: {len(df_families):,} DOCDB families."
    )

    print("\nApplicant extraction:")

    q_applicants = (
        db.query(
            TLS201_APPLN.docdb_family_id,
            TLS201_APPLN.earliest_filing_year,
            TLS206_PERSON.person_id,
            TLS206_PERSON.person_name,
            TLS206_PERSON.psn_name,
            TLS206_PERSON.han_name,
            TLS206_PERSON.psn_sector,
            TLS206_PERSON.person_ctry_code,
        )
        .select_from(TLS201_APPLN)
        .join(
            TLS207_PERS_APPLN,
            TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id,
        )
        .join(
            TLS206_PERSON,
            TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id,
        )
        .filter(TLS201_APPLN.docdb_family_id.in_(family_ids))
        .filter(TLS201_APPLN.ipr_type == IPR_TYPE)
        .filter(
            TLS201_APPLN.earliest_filing_year.between(
                YEAR_MIN, YEAR_MAX
            )
        )
        .filter(TLS207_PERS_APPLN.applt_seq_nr > 0)
        .distinct()
    )

    df_appl = pd.DataFrame(patstat_client.df(q_applicants))

    if df_appl.empty:
        print("Warning: no applicants found for the selected families.")

        return df_families, pd.DataFrame(
            columns=[
                "docdb_family_id",
                "earliest_filing_year",
                "person_id",
                "person_name",
                "psn_name",
                "han_name",
                "psn_sector",
                "person_ctry_code",
                "clean_applicant",
                "harmonized_applicant",
                "is_truncated",
            ]
        )

    df_appl["clean_applicant"] = (
    df_appl["psn_name"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna(
        df_appl["han_name"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )
    .fillna(
        df_appl["person_name"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )
    .fillna("")
    .str.upper()
)

    df_appl["harmonized_applicant"] = (
        df_appl["clean_applicant"]
        .apply(harmonize_applicant)
    )

    df_appl_clean = (
        df_appl
        .drop_duplicates(
            subset=[
                "docdb_family_id",
                "harmonized_applicant",
            ]
        )
        .reset_index(drop=True)
    )

    df_appl_clean["is_truncated"] = (
        df_appl_clean[FAMILY_TIME_COLUMN] > LAST_COMPLETE_YEAR
    )

    print(
        f"Cleaned applicant dataset: "
        f"{len(df_appl_clean):,} family-applicant records."
    )

    return df_families, df_appl_clean


if __name__ == "__main__":

    print(
        f"Environment: {ENV} | "
        f"Years: {YEAR_MIN}-{YEAR_MAX} | "
        f"Last complete year: {LAST_COMPLETE_YEAR}"
    )

    patstat = PatstatClient(env=ENV)
    db = patstat.orm()

    family_ids = load_family_ids()

    df_families, df_appl_clean = run_cleaning(
    family_ids=family_ids,
    db=db,
    patstat_client=patstat,
    )

    df_families.to_csv(
        OUTPUT_DIR / "families_clean.csv",
        index=False,
    )

    df_appl_clean.to_csv(
        OUTPUT_DIR / "applicants_clean.csv",
        index=False,
    )

   
    print(f"Outputs saved in: {OUTPUT_DIR.resolve()}")


Environment: TEST | Years: 2000-2025 | Last complete year: 2022
Loaded 460 unique DOCDB families.

Patent metadata extraction:
Retrieved 484 patent application records.
Cleaned patent dataset: 378 DOCDB families.

Applicant extraction:
Cleaned applicant dataset: 554 family-applicant records.
Outputs saved in: /home/jovyan/innovation-management-datascience26/outputs/step1


# Step 0 — Patent Dataset Cleaning

## 1. Purpose and overview

This script implements the cleaning phase of Step 0 of the project. It takes the DOCDB family IDs identified during the extraction phase (`01_extract.py`) and retrieves the corresponding patent metadata and applicant information from PATSTAT.

The main objective is to prepare two datasets for the subsequent analyses:

- **Patent-family dataset (`families_clean.csv`)**: one representative record per DOCDB family, including patent metadata, title, abstract and text-preprocessing fields.
- **Applicant dataset (`applicants_clean.csv`)**: applicant records associated with each DOCDB family, including standardized applicant names and a truncation flag.

The cleaning procedure follows the main methodological requirements described in Section 7.3 of the *PW Explanatory Guide*: using DOCDB families as the unit of analysis, harmonizing applicant names and identifying potentially incomplete recent observations.

The implementation also incorporates the text-processing instructions reported in Tutorial 2, Notebooks 20 and 21.

## 2. Configuration and input data

The script starts by defining the main configuration parameters:

- `ENV = "TEST"` selects the reduced PATSTAT database, which is recommended during development. The complete database (`PROD`) should be used when producing the final project results.
- `YEAR_MIN` and `YEAR_MAX` define the observation period (2000–2025).
- `FAMILY_TIME_COLUMN` and `APPLN_TIME_COLUMN` specify the time variables used in the analysis.
- `TRUNCATED_YEARS` defines the number of recent years to be flagged as potentially incomplete for the selected time variables.
- `LAST_COMPLETE_YEAR` calculates the last year considered complete according to the configured truncation window.
- `IPR_TYPE = "PI"` restricts the queries to patent inventions.
- `INPUT_FILE` identifies the CSV generated by `01_extract.py`, containing the selected DOCDB family IDs.
- `OUTPUT_DIR` identifies the directory in which the cleaned datasets are saved.

The `load_family_ids()` function reads and validates the input file. It checks that the required column exists, converts the identifiers into numeric values, removes missing and duplicate IDs, and returns a list of unique family identifiers.

This separation between extraction and cleaning ensures that the cleaning script works on the families already selected during the sector-identification phase, without repeating the original sector query.

**Sources and implementation:**
- *PW Explanatory Guide*, Section 7.4, for the distinction between TEST and PROD environments.
- Tutorial 0, Notebook 1 (*PATSTAT on TIP*), for PATSTAT client initialization and database access.


## 3. Patent metadata extraction

### 3.1 PATSTAT tables and query construction

The `run_cleaning()` function retrieves patent application information using the PATSTAT SQLAlchemy ORM.

The function receives three arguments:

- `family_ids`: the list of DOCDB family identifiers obtained from the extraction phase.
- `db`: the PATSTAT ORM database object, used to construct the queries.
- `patstat_client`: the PATSTAT client, used to execute the queries and convert the results into Pandas DataFrames.

Passing the client explicitly to the function avoids relying on a global variable and makes the function easier to reuse independently of the main execution block.

The first query combines the following tables:

| PATSTAT table | Information retrieved |
|---|---|
| `TLS201_APPLN` | Application identifiers, DOCDB family IDs, filing years, application authorities, family size and citation information |
| `TLS202_APPLN_TITLE` | Patent application titles |
| `TLS203_APPLN_ABSTR` | Patent abstracts and abstract language codes |

The tables are connected through `appln_id`, which identifies individual patent applications.

The query is restricted to the DOCDB family IDs obtained from the extraction phase, the selected IPR type and the configured time interval. Outer joins are used for titles and abstracts so that applications are not automatically discarded when one of these text records is unavailable.

The query is executed through `patstat_client.df()` and converted into a Pandas DataFrame.

**Source:** Tutorial 0, Notebook 1 (*PATSTAT on TIP*), introduces the PATSTAT client, ORM query construction, table models, joins and `patstat.df()`. The specific combination of columns and filters used here is adapted to the requirements of this project.

### 3.2 English abstract identification

The script creates the Boolean variable `is_en_abstract` to identify applications with an English abstract suitable for subsequent text-based analyses.

The identification is based on the abstract language field (`appln_abstract_lg`) and the availability of the abstract text.

This distinction is important because patent abstracts may be available in different languages, whereas the subsequent NLP analyses are designed to use English-language documents.

The original abstract and its language code are retained in the dataset, allowing records without an English abstract to remain available for non-textual analyses.

**Source:** `Tutorial 2

### 3.3 DOCDB family deduplication

The DOCDB family is the unit of analysis adopted for the project. A family groups applications associated with the same invention, potentially filed in different patent offices. Counting each application separately could therefore count the same invention multiple times.

To obtain one representative record per family, the script:

1. Sorts the application records by `docdb_family_id`.
2. Prioritizes records with an available English abstract.
3. Among records with the same abstract availability, prioritizes the earliest `appln_filing_year`.
4. Keeps only the first record for each DOCDB family using `drop_duplicates()`.

The result is a family-level dataset containing one representative application record per DOCDB family.

The preference for English abstracts and the selection of the earliest application filing year are implementation choices made by the project team to select the representative record. They are not presented as a specific mandatory sorting procedure in the Explanatory Guide.

**Sources:**
- *PW Explanatory Guide*, Section 7.3, explicitly establishes the DOCDB family as the unit of analysis, so that each invention is counted once.
`drop_duplicates("docdb_family_id")` in Tutorial 2, Notebook 21, Cell 3.

### 3.4 Truncation flag

Patent databases contain incomplete information for recent years because patent applications are generally published after an approximately 18-month delay.

For this reason, the project requires recent observations to be flagged rather than interpreted as evidence of a decline in patenting activity.

The script defines a `TRUNCATED_YEARS` configuration and calculates `LAST_COMPLETE_YEAR` using the maximum truncation window associated with the selected family-level and application-level time variables.

With the current configuration:

- Observation period: 2000–2025.
- Truncation window: 3 years.
- Last complete year: 2022.

The Boolean variable `is_truncated` identifies records whose `earliest_filing_year` is later than the last complete year.

The same truncation criterion is applied to the applicant dataset, allowing recent patent-family and applicant observations to be identified consistently.

**Sources:**
- *PW Explanatory Guide*, Section 7.3, which explains the 18-month publication lag and recommends flagging approximately the last three years.


### 3.5 Abstract preparation and text preprocessing

The script creates two additional text fields:

- `abstract_en`: stores the original abstract when it is available in English, preserving the text before NLP preprocessing.
- `clean_abstract`: contains the preprocessed version of the abstract, intended for subsequent text-based analyses.

The preprocessing is implemented through two functions.

**`strip_tags(text)`**

This function removes XML/HTML-like tags and normalizes whitespace. It replaces tag expressions with spaces and collapses consecutive whitespace characters.

The function is based on the implementation presented in Tutorial 2, Notebook 20, in the code cell defining `strip_tags()`.

In the original notebook, the function is used to process full-text patent information retrieved through EPAB, where XML tags may appear in fields such as titles, abstracts, claims and descriptions. In the present script, the same function is reused as a preliminary text-cleaning operation on PATSTAT abstracts.

**`clean_patent_text(text)`**

This function prepares the abstract for subsequent NLP analyses through the following operations:

1. Handles missing or empty text.
2. Removes tags and normalizes whitespace through `strip_tags()`.
3. Converts the text to lowercase.
4. Extracts alphabetic tokens using a regular expression, allowing internal hyphens.
5. Removes general English stopwords and patent-specific stopwords.
6. Reconstructs the remaining tokens into a cleaned text string.

The patent-specific stopword set (`PATENT_STOP_WORDS`) includes recurrent legal and drafting expressions in patent documents, such as *wherein*, *thereof*, *comprising* and *embodiment*. It is combined with the general English stopword list provided by scikit-learn (`ENGLISH_STOP_WORDS`).

The use of patent-specific stopwords follows Tutorial 2, Notebook 21, where this list is introduced for text vectorization and NLP applications. The notebook uses stopwords in the TF-IDF and other text-processing procedures to reduce the influence of recurrent patent-language expressions.

**Sources and adaptations:**
- Tutorial 2, Notebook 21, Cell 7, as the source of `PATENT_STOP_WORDS`.
- Tutorial 2, Notebook 20, `20_ep_fulltext_epab.ipynb`, contains the `strip_tags(text)` function in the code cell dedicated to extracting and cleaning XML-tagged EPAB text.
- Tutorial 2, Notebook 21, `21_encoding_patent_text.ipynb`, provides the methodological reference for tokenization and patent-specific stopwords.
- The combination of general English stopwords, regular-expression tokenization and the complete `clean_patent_text()` pipeline is an implementation of the project team, rather than a procedure explicitly prescribed in the Explanatory Guide.

The original abstract is preserved separately from the cleaned version, allowing subsequent analyses to use the appropriate representation of the text.

## 4. Applicant extraction and harmonization

### 4.1 PATSTAT tables and applicant identification

The second query retrieves applicant information by joining three PATSTAT tables:

| PATSTAT table | Information retrieved |
|---|---|
| `TLS201_APPLN` | DOCDB family ID and earliest filing year |
| `TLS207_PERS_APPLN` | Relationship between applications and persons |
| `TLS206_PERSON` | Applicant names, person identifiers and country information |

The query uses `applt_seq_nr > 0` to select applicant relationships, excluding records that are not identified as applicants.

The following applicant-related fields are retrieved:

- `person_id`: PATSTAT person identifier.
- `person_name`: original person name.
- `psn_name`: PATSTAT standardized name.
- `han_name`: harmonized applicant name.
- `psn_sector`: PATSTAT sector information.
- `person_ctry_code`: applicant country code.

The `.distinct()` operation removes identical records returned by the query.

**Sources:**
- *PW Explanatory Guide*, Section 7.2, identifies `TLS206_PERSON` and `TLS207_PERS_APPLN` as the main tables for applicant information.
- Tutorial 0, Notebook 1, introduces the relationship between these tables and the use of `applt_seq_nr > 0` to identify applicants.
- 

### 4.2 Applicant-name selection

The script creates a unified field called `clean_applicant`, following this priority order:

1. `psn_name`, when available.
2. `han_name`, when the PSN name is unavailable.
3. `person_name`, when neither of the previous fields is available.

Before selecting the next available field, the script converts the candidate name to a string, removes leading and trailing whitespace and replaces empty strings with missing values.

This step is important because a field containing only whitespace should not be considered a valid applicant name. By removing whitespace before applying the fallback logic, the script ensures that an empty PSN name does not prevent the use of an available HAN or original person name.

After the fallback procedure, the selected name is converted to uppercase. Missing values are handled separately by the subsequent standardization function.

The priority given to PSN follows the PATSTAT examples in Tutorial 0, which explain that standardized applicant names generally provide better results than the original `person_name`.

### 4.3 Basic name standardization

The `harmonize_applicant()` function performs an additional standardization step:

- Missing or empty names are replaced with `"UNKNOWN"`.
- Consecutive whitespace characters are collapsed.
- Leading and trailing whitespace is removed.
- Names are converted to uppercase.

The resulting field, `harmonized_applicant`, is used to identify duplicate applicant records within each DOCDB family.

**Important limitation:** this function performs basic textual standardization only. It does not identify different names belonging to the same legal entity, nor does it consolidate subsidiaries with their parent companies. The Explanatory Guide explicitly requires the use of PSN/HAN fields together with additional rules for subsidiaries. This corporate-group consolidation remains a separate task to be implemented and documented.

**Sources:**
- Tutorial 0, Notebook 1, applicant query examples, for the use of `psn_name` instead of relying exclusively on `person_name`.
- *PW Explanatory Guide*, Section 7.3, for the requirement to harmonize applicant names using PSN/HAN and additional subsidiary rules.


### 4.4 Family-applicant deduplication

After standardizing applicant names, the script removes duplicate combinations of:

- `docdb_family_id`
- `harmonized_applicant`

This ensures that the same standardized applicant is represented only once within a given DOCDB family.

The resulting dataset has a family-applicant structure: a single family can be associated with multiple applicants, but each family-applicant pair appears only once.

The procedure does not merge different applicants across different families, nor does it perform corporate-group consolidation.

### 4.5 Applicant truncation flag

The applicant dataset also receives an `is_truncated` variable, calculated using the same time threshold as the family-level dataset.

This allows recent applicant records to be identified consistently with the patent-family observations.

## 5. Outputs

At the end of the execution, the script saves two CSV files in `outputs/step1/`:

| Output file | Unit of observation | Main contents |
|---|---|---|
| `families_clean.csv` | One row per DOCDB family | Application metadata, title, original abstract, English abstract, cleaned abstract and truncation flag |
| `applicants_clean.csv` | One row per DOCDB family-applicant pair | Applicant identifiers, original and standardized names, country information and truncation flag |

These two datasets provide the cleaned patent-family and applicant information required by the subsequent project steps.

### 5.1 Results of the current execution

Using the PATSTAT TEST environment and the current configuration, the script produced the following results:

- **460 unique DOCDB families** were loaded from the extraction output.
- **484 patent application records** were retrieved from PATSTAT.
- After filtering and family-level deduplication, the cleaned patent dataset contains **378 unique DOCDB families**.
- The applicant extraction produced **554 family-applicant records**, corresponding to **436 unique standardized applicant names**.
- No duplicate DOCDB family records or duplicate family-applicant pairs were found.

Regarding text availability, the subsequent diagnostic checks identified **375 non-empty English abstracts out of 378 families**. Three families do not have an available English abstract: one has a missing language code, one has a French abstract and one has a Chinese abstract.

These families are retained in the general patent-family dataset, but should be excluded from analyses requiring non-empty English text.

The temporal distribution ranges from 2000 to 2023, with 9 families flagged as potentially truncated according to the configured threshold.

These results describe the current execution in the TEST environment and may change when the script is run against the complete PROD database.

## 6. Main methodological references

The cleaning pipeline is based on the following project materials:

1. **PW Explanatory Guide — Section 7.3, "Clean"**: establishes the DOCDB family as the unit of analysis, identifies applicant-name harmonization as a requirement and explains the treatment of recent incomplete years.
2. **PW Explanatory Guide — Section 7.4, "The TIP platform"**: describes the TEST and PROD PATSTAT environments and their intended use.
3. **Tutorial 0 — Notebook 1, "PATSTAT on TIP"**: provides the PATSTAT client setup, SQLAlchemy ORM query structure, relevant PATSTAT table models and applicant-query examples.
4. **Tutorial 2 — Notebook 20, `20_ep_fulltext_epab.ipynb`**: provides the `strip_tags(text)` implementation for removing XML tags from full-text patent information retrieved through EPAB.
5. **Tutorial 2 — Notebook 21, `21_encoding_patent_text.ipynb`**: provides the reference for family-level deduplication and patent-specific stopwords used in NLP applications.



# DEBUG

In [5]:
import pandas as pd

families = pd.read_csv("outputs/step1/families_clean.csv")
applicants = pd.read_csv("outputs/step1/applicants_clean.csv")

print("FAMILIES")
print("Number of rows:", len(families))
print("Unique DOCDB families:", families["docdb_family_id"].nunique())
print("Duplicate families:", families["docdb_family_id"].duplicated().sum())

print("\nMISSING VALUES")
print(families.isna().sum())

print("\nABSTRACTS")
print("English abstracts:", (families["abstract_en"].str.strip() != "").sum())
print("Empty abstracts:", (families["abstract_en"].str.strip() == "").sum())

print("\nYEARS")
print(families["earliest_filing_year"].describe())
print("Truncated families:", families["is_truncated"].sum())

print("\nAPPLICANTS")
print("Number of records:", len(applicants))
print("Unique applicants:", applicants["harmonized_applicant"].nunique())
print("Missing applicant names:", (applicants["harmonized_applicant"] == "UNKNOWN").sum())

print("\nDUPLICATE FAMILY-APPLICANT PAIRS")
print(
    applicants.duplicated(
        subset=["docdb_family_id", "harmonized_applicant"]
    ).sum()
)


FAMILIES
Number of rows: 378
Unique DOCDB families: 378
Duplicate families: 0

MISSING VALUES
docdb_family_id         0
appln_id                0
appln_auth              0
appln_filing_year       0
earliest_filing_year    0
docdb_family_size       0
nb_citing_docdb_fam     0
appln_title             0
appln_abstract          1
appln_abstract_lg       1
is_en_abstract          0
is_truncated            0
abstract_en             3
clean_abstract          3
dtype: int64

ABSTRACTS
English abstracts: 378
Empty abstracts: 0

YEARS
count     378.000000
mean     2016.235450
std         5.363212
min      2000.000000
25%      2013.000000
50%      2018.000000
75%      2020.000000
max      2023.000000
Name: earliest_filing_year, dtype: float64
Truncated families: 9

APPLICANTS
Number of records: 554
Unique applicants: 436
Missing applicant names: 0

DUPLICATE FAMILY-APPLICANT PAIRS
0


In [6]:
abstracts = families["abstract_en"].fillna("").astype(str).str.strip()

print("Non empty Abstract:", abstracts.ne("").sum())
print("Empty Abstract:", abstracts.eq("").sum())

print("\nFamilies without abstract:")
print(
    families.loc[
        abstracts.eq(""),
        ["docdb_family_id", "appln_title", "appln_abstract_lg"]
    ]
)

Non empty Abstract: 375
Empty Abstract: 3

Families without abstract:
     docdb_family_id                                        appln_title  \
6           32476259                             Unmanned ocean vehicle   
106         53794250  MODULE SOUS-MARIN DE PRODUCTION D'ENERGIE ELEC...   
377         89206667                                一种海洋科学试验用海洋剖面主动观测装置   

    appln_abstract_lg  
6                 NaN  
106                fr  
377                zh  


## Debugging and Data Quality Checks

The debugging checks are performed to verify the quality, consistency, and completeness of the datasets generated by `02_clean.py` before using them in the subsequent analyses.

### 1. Families validation

The first check verifies that the `families_clean.csv` dataset contains exactly one record for each DOCDB family. It compares the total number of rows with the number of unique family identifiers and counts possible duplicates. This is important because the DOCDB family is the unit of analysis adopted for the project.

**Result:** 378 records, 378 unique DOCDB families, and no duplicate families.

### 2. Missing values

This check counts missing values (`NaN`) in each column of the families dataset. Its purpose is to identify potentially incomplete patent records and verify that the main metadata fields have been successfully retrieved from PATSTAT.

**Result:** the main metadata fields contain no missing values. Missing values are present only in the abstract-related columns.

### 3. Abstract availability

This check aims to count the English abstracts available for the subsequent text-based analyses and identify records without an English abstract.

The initial check uses `.str.strip()` to remove leading and trailing spaces before comparing values with an empty string. However, it does not explicitly handle `NaN` values. Since missing values are not treated as empty strings, the initial output incorrectly reports 378 English abstracts and zero empty abstracts.

To obtain the correct count, missing values are first replaced with empty strings using `.fillna("")`, and the values are converted to strings and stripped of whitespace.

**Correct result:** 375 non-empty English abstracts and 3 empty abstracts.

### 4. Year distribution and truncated families

The `describe()` function provides descriptive statistics for `earliest_filing_year`, including the minimum, maximum, mean, median, and quartiles. This helps inspect the temporal distribution of the patent families and identify possible anomalies.

The number of truncated families is also calculated through the `is_truncated` flag. This check is relevant because the most recent years may be affected by the 18-month patent publication delay and therefore contain incomplete information.

**Result:** the dataset covers the period 2000–2023, and 9 families are marked as truncated.

### 5. Applicant validation

This check examines the `applicants_clean.csv` dataset by counting the total number of family-applicant records and the number of unique harmonized applicant names. It also counts records whose applicant name is equal to `UNKNOWN`, which is used as a placeholder when no applicant name can be retrieved.

**Result:** 554 family-applicant records, 436 unique applicants, and no applicant names marked as `UNKNOWN`.

### 6. Duplicate family-applicant pairs

The final check verifies that the same harmonized applicant is not associated more than once with the same DOCDB family. It counts duplicate combinations of `docdb_family_id` and `harmonized_applicant`.

This is important to ensure that each family-applicant relationship is represented only once in the final dataset.

**Result:** no duplicate family-applicant pairs were found.

### 7. Identification of families without an English abstract

The final diagnostic check provides further information about the three families without a non-empty English abstract. It identifies these records and displays their DOCDB family ID, patent title, and original abstract language, when available.

The results show three different cases:
- One family has no available abstract language information (`NaN`).
- One family has an abstract in French (`fr`).
- One family has an abstract in Chinese (`zh`).

These families are retained in the general patent dataset, but their abstracts cannot be used in the subsequent English-language NLP analyses.

### Overall purpose

Overall, these debugging checks provide a quality-control step for the cleaning pipeline. They verify family-level uniqueness, metadata completeness, abstract availability, temporal consistency, applicant-name quality, and the absence of duplicate family-applicant relationships. This helps ensure that the cleaned datasets are reliable and suitable for the subsequent analyses.